# Legal Analysis Evaluation

## Purpose

This notebook is designed to evaluate how well the platform can perform legal analysis when operating under conditions that replicate its intended real-world use. In practice, the platform will be asked to analyze a contract dispute before it reaches a court or board decision. To simulate that scenario, the notebook begins with full appellate case PDFs and produces a pre-appeal version of each document by removing all judicial reasoning, orders, and litigation-stage narrative.

The platform then performs its facts and rules extraction on this simulated pre-appeal text. Its results are compared against reference annotations that were generated from the *full* court decision, which contain more complete, court-informed reasoning. This allows users to measure how closely the platform’s pre-appeal analysis matches what is later reflected in the court’s findings.

The platform then performs two forms of evaluation:

1. **Extraction matching** – comparing its facts and rules extractions from the pre-appeal text to reference annotations generated from the full court decision.
2. **Scoring comparison** – comparing legal reasoning scores from the platform’s scoring endpoint for both the complete decision and the pre-appeal version, to measure the impact of removing judicial analysis.

## Inputs

* **Case PDFs** – Appellate case documents that include both the underlying factual record and the court or board’s decision and orders.
* **Reference annotations (ChromaDB)** – Structured data for each case, containing:

  * Directly quoted facts from the record,
  * Procedural rules governing how the claim was processed,
  * Substantive rules defining legal rights and entitlements,
  * A structured Q\&A summarizing case background.
    These annotations are derived from the complete appellate decision and thus incorporate judicial reasoning and conclusions.

* **Models and services** –

  * A language model to produce the simulated pre-appeal text,
  * The platform’s API endpoint to extract facts, procedural rules, and substantive rules,
  * An embeddings model to compare the platform’s output to the reference annotations.

## Workflow

**Step 1 – Convert PDF to text**
Each appellate case PDF is converted to structured text using `pymupdf4llm`. This ensures consistent formatting and preserves the original content for processing.

**Step 2 – Create simulated pre-appeal text**
The converted text is processed with the `CREATE_PRE_APPEAL_DOCUMENT_PROMPT` to remove:

* Judicial analysis and reasoning by the court or board,
* Procedural developments occurring during the appeal,
* Any headings or labels summarizing the appeal outcome,
* Content introduced during litigation.
  The result is a document that reflects only the facts, contracting officer decisions, and related correspondence available before any appeal was filed.

**Step 3 – Run the platform’s extraction endpoint**
The simulated pre-appeal text is sent to the platform’s API, which applies the remaining prompt workflows to extract:

* **Facts** – Direct quotations from the record,
* **Procedural rules** – Requirements and processes for claim handling prior to appeal (e.g., jurisdiction, certification, deadlines, contracting officer authority, issuance of a final decision),
* **Substantive rules** – Legal doctrines and contract clauses governing entitlements (e.g., Suspension of Work, Changes, Eichleay formula).

**Step 4 – Retrieve reference annotations from ChromaDB**
For each case, the notebook retrieves the reference facts, procedural rules, and substantive rules from the ChromaDB dataset. These were built with full access to the court’s decision and incorporate the legal reasoning found in that decision.

**Step 5 – Compare platform outputs to reference annotations**
The comparison process includes:

* Text normalization (lowercasing, trimming whitespace),
* Generating embeddings for both platform outputs and reference items,
* Calculating cosine similarity between every possible pair,
* Using the Hungarian algorithm to find a one-to-one matching that maximizes total similarity,
* Counting a pair as a match if similarity meets or exceeds a set threshold (e.g., 0.85).

**Step 6 – Score results**
For each category (facts, procedural rules, substantive rules) and each case, the notebook calculates:

* **Precision** – Of the items extracted by the platform, the proportion that matched the reference annotations,
* **Recall** – Of the reference annotations, the proportion found by the platform,
* **F1 score** – The harmonic mean of precision and recall.
  Scores are reported per case and as macro-averages across all cases. The notebook can also display side-by-side listings of matches and non-matches for review.

**Step 7 – Compare legal reasoning scores**
The notebook calls the platform’s **scoring\_results** endpoint for:

* The **complete** appellate document, and
* Its **pre-appeal** counterpart.

For each, it retrieves:

* `total_admissibility_score` – a composite measure of how the platform’s analysis addresses procedural viability,
* `total_relevance_score` – a composite measure of how the platform’s analysis aligns with substantive legal relevance.

A side-by-side table is printed for each case, showing both scores under each document condition.

## Purpose of the Pre-Appeal Simulation

The pre-appeal simulation step ensures that the platform is evaluated in the same informational environment in which it is intended to operate: with access only to the record content available before litigation, and without the benefit of court or board reasoning.

By comparing this “pre-appeal” analysis to reference annotations informed by the full decision, stakeholders can assess:

* How effectively the platform can reconstruct key facts and rules from limited information,
* Where it aligns with court-level conclusions,
* Where gaps exist that may require more reasoning capability or additional context.

## Legal Significance of Each Output Type

* **Facts** – Statements from the record that could form the basis for legal arguments or findings of fact in an eventual decision.
* **Procedural rules** – Process requirements that can determine whether a claim is valid or can proceed (e.g., jurisdictional prerequisites, timeliness, certification). These affect admissibility and procedural outcomes.
* **Substantive rules** – Legal principles and contract provisions that determine rights, entitlements, and remedies. These affect the merits of a dispute.


## Interpreting the Metrics

* **Precision** reflects the proportion of correct extractions relative to all extractions made by the platform.
* **Recall** reflects the proportion of correct extractions made by the platform relative to all relevant items in the reference annotations.
* **F1 score** balances these two measures to give a single indicator of performance.

Because the reference annotations are based on full court decisions, some items may depend on reasoning not explicitly stated in the pre-appeal text. In such cases, a “miss” does not necessarily indicate an error by the platform, but rather a limitation in the available input.


## Imports and Schemas

In [24]:
import os
import json
import glob
import requests
import chromadb
from typing import TypedDict
import numpy as np
import pymupdf4llm
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from dotenv import load_dotenv

load_dotenv()

BASE_DATA_PATH = os.path.join(os.getcwd(), "../data")
CHROMA_PATH = os.path.join(os.getcwd(), "../extraction_chroma_db")
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_collection("contract_disputes_chunks")

model = ChatOpenAI(model=os.getenv("OPENAI_MODEL"), temperature=0)


class CaseEvalResult(TypedDict):
    precision: float
    recall: float
    f1: float
    true_pos: int
    false_pos: int
    false_neg: int


class Fact(TypedDict):
    id: int
    specific_fact_cited: str
    relevance_reason: str
    contestability_reason: str


class ProceduralRule(TypedDict):
    procedural_rule: str
    effects: str


class SubstantiveRule(TypedDict):
    substantive_law: str
    applicability: str
    relevance: str


class FactsExtractionOutput(TypedDict):
    facts: list[Fact]


class ProceduralRulesOutput(TypedDict):
    rules: list[ProceduralRule]


class SubstantiveRulesOutput(TypedDict):
    rules: list[SubstantiveRule]


class ChromaFactsAndRules(TypedDict):
    qa: str
    facts: list[Fact]
    procedural_rules: list[ProceduralRule]
    substantive_rules: list[SubstantiveRule]


class FactsAndRulesOutput(TypedDict):
    facts: FactsExtractionOutput
    procedural_rules: ProceduralRulesOutput
    substantive_rules: SubstantiveRulesOutput


## Prompts

In [ ]:
CREATE_PRE_APPEAL_DOCUMENT_PROMPT = """
From the following document (which may be a court decision, order, or dismissal), generate a new version that contains only the content that was available before the appeal was filed.

Preserve the original content exactly as written, including formatting, names, dates, quotations, and structure. Do not rewrite, summarize, or paraphrase any portion of the text.

Your only task is to remove content that falls into any of the following categories:

- Legal reasoning, analysis, findings, or citations by the Board or court
- Procedural history or developments that occurred after the appeal was filed
- Statements, footnotes, or narrative from the Board describing or resolving the appeal
- Any label, heading, or caption that states or summarizes the result of the appeal or any procedural rulings made during the appeal. This includes, but is not limited to:
  - “DENIED: [date]”
  - “DISMISSED WITH PREJUDICE: [date]”
  - “DISMISSED IN PART FOR LACK OF JURISDICTION: [date]”
  - “GRANTED IN PART: [date]”
  - “APPELLANT’S MOTION FOR SUMMARY JUDGMENT DENIED; RESPONDENT’S MOTION FOR SUMMARY JUDGMENT GRANTED IN PART: [date]”
  - Any similar summary or caption reflecting a Board or court decision, procedural disposition, or ruling on motions
- Any content that was introduced during or after the litigation or decision-making process

The output must read as a complete, self-contained document that existed prior to litigation, reflecting only the parties’ claims, the contracting officer’s actions and decisions, and related correspondence.

Do not include any commentary, explanations to the reader, or editorial statements.

Input:
DOCUMENT TEXT:
{text}
"""

## Functions

In [30]:
def call_rules_extraction(md_text, base_url):
    url = f"{base_url}/api/legal_analysis/rules_extraction"
    resp = requests.post(url, params={"md_text": md_text})
    resp.raise_for_status()
    return resp.json()


def call_scoring_results(md_text, base_url):
    url = f"{base_url}/api/legal_analysis/scoring_results"
    resp = requests.post(url, params={"md_text": md_text})
    resp.raise_for_status()
    return resp.json()


def normalize_text(text: str) -> str:
    """Lowercase, trim, and collapse spaces for consistent matching."""
    return " ".join(text.lower().strip().split())


def embed_texts(texts: list[str]) -> np.ndarray:
    """Embed a list of strings with LangChain OpenAIEmbeddings, L2-normalized."""
    if not texts:
        raise Exception("Empty list of texts")
    embeddings = OpenAIEmbeddings(model=os.getenv("OPENAI_EMBEDDING_MODEL"))
    vectors = np.array(embeddings.embed_documents(texts), dtype=np.float32)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)  # + 1e-12
    return vectors / norms


def cosine_sim_matrix(
    left_vectors: np.ndarray, right_vectors: np.ndarray
) -> np.ndarray:
    """Compute cosine similarity for L2-normalized vectors."""
    if left_vectors.size == 0 or right_vectors.size == 0:
        return np.zeros(
            (left_vectors.shape[0], right_vectors.shape[0]), dtype=np.float32
        )
    return left_vectors @ right_vectors.T


def evaluate_llm_response_items(
    case_data: list[str], response_items: list[str], threshold: float
) -> CaseEvalResult:
    """
    Evaluate the items generated by the LLM's response (facts or rules) with respect the reference case_data.

    Steps:
      1) Normalize texts
      2) Embed case_data and response_items
      3) Compute cosine similarities
      4) Hungarian matching (maximize similarity)
      5) Count matches with similarity >= threshold
      6) Return precision, recall, F1, and TP/FP/FN
    """
    # 1) Normalize texts
    case_norm = [normalize_text(x) for x in case_data]
    response_norm = [normalize_text(x) for x in response_items]
    if not response_norm:
        return {
            "precision": 1.0,
            "recall": 0.0,
            "f1": 0.0,
            "true_pos": 0,
            "false_pos": 0,
            "false_neg": len(case_norm),
        }

    # 2) Embed case_data and response_items
    case_emb = embed_texts(case_norm)
    response_emb = embed_texts(response_norm)

    # 3) Compute cosine similarities
    sim_matrix = cosine_sim_matrix(case_emb, response_emb)

    # 4) Hungarian matching (maximize similarity)
    case_idx, resp_idx = linear_sum_assignment(
        -sim_matrix
    )  # Solve the linear sum assignment problem
    pair_sims = sim_matrix[case_idx, resp_idx]

    # 5) Count matches with similarity >= threshold
    # 2 masks, one for case and one for response
    matched_mask = pair_sims >= threshold
    matched_case_indices = {
        case_idx[k] for k in range(len(case_idx)) if matched_mask[k]
    }
    matched_resp_indices = {
        resp_idx[k] for k in range(len(resp_idx)) if matched_mask[k]
    }

    # 6) Return precision, recall, F1, and TP/FP/FN
    true_pos = len(matched_resp_indices)
    false_pos = len(response_norm) - true_pos
    false_neg = len(case_norm) - len(matched_case_indices)

    precision = true_pos / len(response_norm)
    recall = true_pos / len(case_norm)
    f1 = (
        (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_pos": true_pos,
        "false_pos": false_pos,
        "false_neg": false_neg,
    }


def create_pre_appeal_document(md_text: str) -> str:
    """
    Generates a new version of the document that contains only the content that
    was available before the appeal was filed.
    """
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", CREATE_PRE_APPEAL_DOCUMENT_PROMPT),
            ("user", "{text}"),
        ]
    )
    chain = prompt | model | StrOutputParser()
    return chain.invoke({"text": md_text})


def facts_and_rules_tables(
    pdf_files: list[str], base_url: str
) -> tuple[dict[str, ChromaFactsAndRules], dict[str, FactsAndRulesOutput]]:
    """Extracts generates the tables"""
    facts_and_rules_dataset = {}
    endpoint_responses = {}
    for doc_path in pdf_files:
        doc_name = os.path.splitext(os.path.basename(doc_path))[0]
        results = collection.get(where={"doc_name": doc_name})
        facts_and_rules_dataset[f"{doc_name}"] = {
            results["metadatas"][i]["type"]: json.loads(results["documents"][i])
            for i in range(4)
        }
        md_text = create_pre_appeal_document(pymupdf4llm.to_markdown(doc_path))
        endpoint_responses[f"{doc_name}"] = call_rules_extraction(md_text, base_url)
    return facts_and_rules_dataset, endpoint_responses


def display_lists(
    doc_name: str,
    facts_and_rules_dataset: dict[str, ChromaFactsAndRules],
    endpoint_responses: dict[str, FactsAndRulesOutput],
    list_type: str,
) -> None:
    """Display lists of facts or rules for comparison (printed separately)."""

    cfg = {
        "facts": (
            "Facts",
            ("facts", "specific_fact_cited"),
            ("facts", "facts", "specific_fact_cited"),
        ),
        "procedural_rules": (
            "Procedural Rules",
            ("procedural_rules", "procedural_rule"),
            ("procedural_rules", "rules", "procedural_rule"),
        ),
        "substantive_rules": (
            "Substantive Rules",
            ("substantive_rules", "substantive_law"),
            ("substantive_rules", "rules", "substantive_law"),
        ),
    }
    if list_type not in cfg:
        raise ValueError(f"Unknown list_type: {list_type!r}")

    title, (ds_key, ds_field), (ep_key, ep_subkey, ep_field) = cfg[list_type]

    ds_items = facts_and_rules_dataset.get(doc_name, {}).get(ds_key, []) or []
    ep_items = (
        endpoint_responses.get(doc_name, {}).get(ep_key, {}).get(ep_subkey, []) or []
    )

    print(f"\n{title}: facts_and_rules_dataset")
    for i, row in enumerate(ds_items, 1):
        print(f"  {i}. {row.get(ds_field, '')}")

    print(f"\n{title}: endpoint_responses")
    for i, row in enumerate(ep_items, 1):
        print(f"  {i}. {row.get(ep_field, '')}")


def evaluation_metrics(
    pdf_files: list[str],
    facts_and_rules_dataset: dict[str, ChromaFactsAndRules],
    endpoint_responses: dict[str, FactsAndRulesOutput],
    threshold: int,
) -> tuple[CaseEvalResult, CaseEvalResult, CaseEvalResult]:
    """Calculate and display precision, recall and F1 score per document."""

    facts_matching_scores: CaseEvalResult = {}
    procedural_rules_matching_scores: CaseEvalResult = {}
    substantive_rules_matching_scores: CaseEvalResult = {}

    # (dataset_key, field, endpoint_key, endpoint_subkey)
    cfg = {
        "facts": ("facts", "specific_fact_cited", "facts", "facts"),
        "procedural_rules": (
            "procedural_rules",
            "procedural_rule",
            "procedural_rules",
            "rules",
        ),
        "substantive_rules": (
            "substantive_rules",
            "substantive_law",
            "substantive_rules",
            "rules",
        ),
    }

    dest = {
        "facts": facts_matching_scores,
        "procedural_rules": procedural_rules_matching_scores,
        "substantive_rules": substantive_rules_matching_scores,
    }

    for doc_path in pdf_files:
        doc_name = os.path.splitext(os.path.basename(doc_path))[0]

        facts_list = [
            row.get("specific_fact_cited")
            for row in facts_and_rules_dataset[doc_name]["facts"]
        ]
        facts_pred = [
            row.get("specific_fact_cited")
            for row in endpoint_responses[doc_name]["facts"]["facts"]
        ]
        dest["facts"][doc_name] = evaluate_llm_response_items(
            facts_list, facts_pred, threshold
        )

        proc_list = [
            row.get("procedural_rule")
            for row in facts_and_rules_dataset[doc_name]["procedural_rules"]
        ]
        proc_pred = [
            row.get("procedural_rule")
            for row in endpoint_responses[doc_name]["procedural_rules"]["rules"]
        ]
        dest["procedural_rules"][doc_name] = evaluate_llm_response_items(
            proc_list, proc_pred, threshold
        )

        subst_list = [
            row.get("substantive_law")
            for row in facts_and_rules_dataset[doc_name]["substantive_rules"]
        ]
        subst_pred = [
            row.get("substantive_law")
            for row in endpoint_responses[doc_name]["substantive_rules"]["rules"]
        ]
        dest["substantive_rules"][doc_name] = evaluate_llm_response_items(
            subst_list, subst_pred, threshold
        )

        print(f"\nDocument {doc_name}, scores for the matching of:")
        print(f"  Facts:             {facts_matching_scores[doc_name]}")
        print(f"  Procedural rules:  {procedural_rules_matching_scores[doc_name]}")
        print(f"  Substantive rules: {substantive_rules_matching_scores[doc_name]}")

    return (
        facts_matching_scores,
        procedural_rules_matching_scores,
        substantive_rules_matching_scores,
    )


def avg_metrics(scores: CaseEvalResult) -> tuple[float, float, float]:
    """Compute average precision, recall, f1 for given scores dict."""
    return tuple(
        np.mean([scores[doc][metric] for doc in scores])
        for metric in ("precision", "recall", "f1")
    )


def display_metrics_avg(
    facts_matching_scores: CaseEvalResult,
    procedural_rules_matching_scores: CaseEvalResult,
    substantive_rules_matching_scores: CaseEvalResult,
) -> None:
    """Displays average precision, recall, and F1 score for facts,
    procedural rules, and substantive rules."""

    categories = {
        "Facts": facts_matching_scores,
        "Procedural rules": procedural_rules_matching_scores,
        "Substantive rules": substantive_rules_matching_scores,
    }

    print("\nCategory             Precision   Recall   F1")
    print("-" * 44)
    for name, scores in categories.items():
        precision, recall, f1 = avg_metrics(scores)
        print(f"{name:<20} {precision:>9.3f} {recall:>8.3f} {f1:>8.3f}")


def scoring_legal_reasoning(pdf_files: list[str], base_url: str):
    """Evaluate potential judicial reasoning by scoring procedural and substantive rules 
    from documents containing the court decision/order and its pre-appeal version"""
    complete_doc_responses = {}
    preappeal_doc_responses = {}
    for doc_path in tqdm(pdf_files, desc="Scoring legal reasoning", unit="file"):
        doc_name = os.path.splitext(os.path.basename(doc_path))[0]
        print(f"processing {doc_name}...")
        md_text = pymupdf4llm.to_markdown(doc_path)
        md_text_pre = create_pre_appeal_document(md_text)
        complete_doc_responses[f"{doc_name}"] = call_scoring_results(md_text, base_url)
        preappeal_doc_responses[f"{doc_name}"] = call_scoring_results(md_text_pre, base_url)
    return complete_doc_responses, preappeal_doc_responses

# Evaluating the match between analysis from the API and reference data

### Get facts and rules from the Chroma facts dataset and the API endpoint 

In [25]:
base_url = "http://127.0.0.1:8000"
year = "2026"
threshold = 0.85
year_folder = os.path.join(BASE_DATA_PATH, year)
pdf_files = glob.glob(os.path.join(year_folder, "*.pdf"))

In [ ]:
facts_and_rules_dataset, endpoint_responses = facts_and_rules_tables(pdf_files, base_url)

### Score for the matching between data in the facts dataset and the analysis generated with the API endpoint

In [3]:
(
    facts_matching_scores, procedural_rules_matching_scores, substantive_rules_matching_scores
) = evaluation_metrics(pdf_files, facts_and_rules_dataset, endpoint_responses, threshold)


Document SULLIVAN_04-04-25_7451__QUALITY_TRUST_INC (DECISION) (1), scores for the matching of:
  Facts:             {'precision': 0.5294117647058824, 'recall': 0.6923076923076923, 'f1': 0.5999999999999999, 'true_pos': 9, 'false_pos': 8, 'false_neg': 4}
  Procedural rules:  {'precision': 0.3333333333333333, 'recall': 0.75, 'f1': 0.46153846153846156, 'true_pos': 3, 'false_pos': 6, 'false_neg': 1}
  Substantive rules: {'precision': 0.8, 'recall': 1.0, 'f1': 0.888888888888889, 'true_pos': 4, 'false_pos': 1, 'false_neg': 0}


### Display average Precision, Recall, and F1 Score

In [5]:
display_metrics_avg(facts_matching_scores, procedural_rules_matching_scores, substantive_rules_matching_scores)


Category             Precision   Recall   F1
--------------------------------------------
Facts                    0.529    0.692    0.600
Procedural rules         0.333    0.750    0.462
Substantive rules        0.800    1.000    0.889


### Vizsualizing data from the facts dataset and the API endpoint for one example

In [18]:
doc_name = list(facts_and_rules_dataset.keys())[0]
print(f"doc_name: {list(facts_and_rules_dataset.keys())[0]}")
list_category = 'substantive_rules'
display_lists(
    doc_name, 
    facts_and_rules_dataset, 
    endpoint_responses, 
    list_category
)

doc_name: SULLIVAN_04-04-25_7451__QUALITY_TRUST_INC (DECISION) (1)

Substantive Rules: facts_and_rules_dataset
  1. FAR 52.242-14 Suspension of Work clause
  2. Constructive Suspension Doctrine
  3. Eichleay Formula for Damages
  4. Reasonableness of Suspension Periods

Substantive Rules: endpoint_responses
  1. Suspension of Work clause (FAR 52.242-14)
  2. Constructive Suspension Doctrine
  3. Eichleay Formula
  4. Elements of proving a Suspension of Work claim
  5. Reasonableness of Suspension Periods


# Comparing the API legal score endpoint results between the original court documents and the pre-appeal documents

### Fetch the legal reasoning scorings from the API endpoint

In [29]:
complete_doc_responses, preappeal_doc_responses = scoring_legal_reasoning(pdf_files, base_url)

### Display the scores

In [ ]:
for doc_path in tqdm(pdf_files, desc="Scoring legal reasoning", unit="file"):
    doc_name = os.path.splitext(os.path.basename(doc_path))[0]
    admissibility_score_complete = complete_doc_responses.get(doc_name).get('total_admissibility_score')
    relevance_score_complete= complete_doc_responses.get(doc_name).get('total_relevance_score')
    admissibility_score_pre = preappeal_doc_responses.get(doc_name).get('total_admissibility_score')
    relevance_score_pre = preappeal_doc_responses.get(doc_name).get('total_relevance_score')
    print("-" * 80)
    print(f"Scores for {doc_name}:")
    print("")
    print("Category           Complete   Pre-Appeal")
    print("-" * 40)
    print(f"Admissibility {admissibility_score_complete:>10} {admissibility_score_pre:>12}")
    print(f"Relevance     {relevance_score_complete:>10} {relevance_score_pre:>12}")
        

Scoring legal reasoning: 100%|██████████| 1/1 [00:00<00:00, 5882.61file/s]

--------------------------------------------------------------------------------
Scores for SULLIVAN_04-04-25_7451__QUALITY_TRUST_INC (DECISION) (1):

Category           Complete   Pre-Appeal
----------------------------------------
Admissibility       0.95          0.8
Relevance           0.74         0.92
